In [1]:
import pandas as pd
df = pd.read_csv("hvstat_africa_data_v1.2.csv")
print(df.head(10).to_string())

         fnid country country_code admin_1 admin_2        product season_name  planting_year  planting_month  harvest_year  harvest_month crop_production_system  qc_flag     area  production      yield
0  AO2008A101  Angola           AO   Bengo    none         Banana        Main           2007              10          2008              4               All (PS)        0   1364.0     19588.0  14.360704
1  AO2008A101  Angola           AO   Bengo    none         Banana        Main           2008              10          2009              4               All (PS)        0   1547.0     24121.0  15.592114
2  AO2008A101  Angola           AO   Bengo    none         Banana        Main           2014              10          2015              4               All (PS)        0  11513.0    331527.0  28.795883
3  AO2008A101  Angola           AO   Bengo    none         Banana        Main           2015              10          2016              4               All (PS)        0  11513.0    331527.0  

In [2]:
ng = df[df["country_code"] == "NG"].copy()
print(len(ng), ng["admin_1"].nunique(), ng["harvest_year"].min(), ng["harvest_year"].max())

print(ng.groupby("product").agg(
    rows=("yield", "size"), states=("admin_1", "nunique"),
    first=("harvest_year", "min"), last=("harvest_year", "max")
).sort_values("rows", ascending=False))

print(ng[["season_name", "planting_month", "harvest_month"]].value_counts().head(20))
print(ng["qc_flag"].value_counts())
print(ng["admin_2"].value_counts().head())

# rows whose area and production repeat the previous year's values
key = ["fnid", "product", "season_name"]
ng = ng.sort_values(key + ["harvest_year"])
prev = ng.groupby(key)[["area", "production"]].shift()
same = (ng["area"] == prev["area"]) & (ng["production"] == prev["production"])
print(same.mean())

9636 37 1999 2023
                       rows  states  first  last
product                                         
Maize                   919      37   1999  2023
Rice                    909      37   1999  2023
Cowpea                  861      37   1999  2023
Cassava                 851      37   1999  2023
Groundnuts (In Shell)   782      36   1999  2023
Yams                    697      30   1999  2023
Taro                    612      33   1999  2021
Sorghum                 536      27   1999  2023
Millet                  512      26   1999  2023
Soybean                 495      27   1999  2021
Tomato                  438      34   2010  2023
Okras                   406      37   2010  2021
Sesame Seed             387      29   1999  2021
Sweet Potatoes          378      36   1999  2009
Cotton                  304      18   1999  2023
Melon                   213      26   1999  2010
Onions                  176      16   2010  2021
Wheat                    84      11   1999  2009
Gi

In [ ]:
ng = df[df["country_code"] == "NG"].copy()
print(ng["product"].unique())                      
ng["crop"] = ng["product"].str.lower()
cer = ng[ng["crop"].isin(["maize", "rice", "sorghum", "millet"])].copy()

print(pd.crosstab(cer["crop"], cer["season_name"]))         
print((cer["planting_year"] == cer["harvest_year"]).mean()) 
print(cer["crop_production_system"].value_counts(dropna=False))
print(cer[["area", "production", "yield"]].isna().sum())
print(((cer["production"] / cer["area"]) - cer["yield"]).abs().describe())  ?
print(cer.groupby("crop")["yield"].describe())           
print(pd.crosstab(cer["qc_flag"], cer["crop"]))

<StringArray>
[               'Banana',               'Cassava',                'Cowpea',
 'Groundnuts (In Shell)',                 'Maize',                 'Melon',
                 'Okras',                  'Rice',               'Sorghum',
        'Sweet Potatoes',                  'Taro',                'Tomato',
                  'Yams',                'Cotton',                'Millet',
                'Onions',           'Sesame Seed',               'Soybean',
                 'Wheat',                'Ginger']
Length: 20, dtype: str
season_name  Wet
crop            
maize        919
millet       512
rice         909
sorghum      536
1.0
crop_production_system
All (PS)    2876
Name: count, dtype: int64
area           1
production    15
yield         16
dtype: int64
count    2.860000e+03
mean     4.302751e-17
std      1.092505e-16
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.776357e-15
dtype: float64
         count      mean     

In [ ]:
# 1. Drop rows with no target
n0 = len(cer)
print(cer[cer["yield"].isna()].groupby("crop").size())   # millet 13, rice 3
clean = cer.dropna(subset=["yield"]).copy()
print("rows:", n0, "->", len(clean), "| dropped:", n0 - len(clean))

# 2. Flag IQR outliers per crop 
g = clean.groupby("crop")["yield"]
q1, q3 = g.transform(lambda s: s.quantile(0.25)), g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
clean["iqr_outlier"] = (clean["yield"] < q1 - 1.5 * iqr) | (clean["yield"] > q3 + 1.5 * iqr)
print(clean.groupby("crop")["iqr_outlier"].sum())

# 3. Do the source's flags agree with ours?
print(pd.crosstab(clean["iqr_outlier"], clean["qc_flag"]))

# 4. Look at the extreme rows
cols = ["crop", "admin_1", "harvest_year", "area", "production", "yield", "qc_flag"]
print(clean.sort_values("yield", ascending=False)[cols].head(10))
print(clean.sort_values("yield")[cols].head(10))

# 5. The still-open question: are repeated rows the flag-2 rows?
key = ["fnid", "crop"]
clean = clean.sort_values(key + ["harvest_year"])
prev = clean.groupby(key)[["area", "production"]].shift()
clean["repeat"] = (clean["area"] == prev["area"]) & (clean["production"] == prev["production"])
print(pd.crosstab(clean["repeat"], clean["qc_flag"]))

crop
millet    13
rice       3
dtype: int64
rows: 2876 -> 2860 | dropped: 16
crop
maize      12
millet     26
rice       62
sorghum    13
Name: iqr_outlier, dtype: int64
qc_flag         0  1    2
iqr_outlier              
False        2572  0  175
True          108  5    0
         crop  admin_1  harvest_year     area  production     yield  qc_flag
151857   rice      Edo          2012  13710.0    129300.0  9.431072        1
154837   rice     Kogi          2011  60920.0    490429.0  8.050378        0
154836   rice     Kogi          2010  60110.0    480804.0  7.998736        0
154838   rice     Kogi          2012  62290.0    492441.0  7.905619        0
157542   rice   Rivers          2012  15590.0    115350.0  7.398974        1
150933  maize    Borno          2012  94000.0    623920.0  6.637447        1
149929   rice  Anambra          2012  15770.0     98200.0  6.227013        1
155429   rice    Lagos          2012   7700.0     43800.0  5.688312        0
150932  maize    Borno          2